In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import json
import pandas as pd
from datasets import load_dataset
import chromadb
from rag_bench import evaluator, helper

from pipeline.ingestion.chroma import insert_data
from pipeline.retriever import ChromaRetriever, BM25Retriever, HybridRetriever
from pipeline.reranker import CrossEncoderReranker

### Загрузка датасетов

In [3]:
CACHE_DIR = "../datasets/"

HIST_PRIVATE_QA_REPO_ID = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID = "ai-forever/hist-rag-bench-private-texts"
PUBLIC_TEXTS_REPO = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO = "ai-forever/hist-rag-bench-public-questions"


def get_public_to_private_texts_mapping(version):
    private_texts_ds = load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version, cache_dir=CACHE_DIR)
    return {item["public_id"]: item["id"] for item in private_texts_ds["train"]}


version = helper.get_latest_version(helper.get_ds_versions(PUBLIC_TEXTS_REPO))
texts_ds = load_dataset(PUBLIC_TEXTS_REPO, revision=version, cache_dir=CACHE_DIR)
qa_dataset = load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version, cache_dir=CACHE_DIR)
mapping = get_public_to_private_texts_mapping(version)
print("Версия датасета:", version)

Версия датасета: 1.15.0


### Индексация

In [4]:
CHROMA_PATH = "../chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

COLLECTION_NAME = "hybrid_bench"

try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

insert_data(
    dataset=texts_ds,
    collection=collection,
    split_type="recursive_character",
    chunk_size=500,
    chunk_overlap=50,
)

Добавлено 2354 чанков в коллекцию 'hybrid_bench'


2354

### Инициализация ретриверов и реранкера

In [5]:
semantic_retriever = ChromaRetriever(collection)
bm25_retriever = BM25Retriever(collection)   # строит BM25-индекс по всему корпусу
hybrid_retriever = HybridRetriever(collection)
reranker = CrossEncoderReranker()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 652.93it/s, Materializing param=classifier.weight]                                      


### Вспомогательные функции

In [6]:
def evaluate_retriever(retriever, qa_dataset, mapping, retrieve_n=3, reranker=None, rerank_max_n=5):
    """Прогоняет все вопросы через retriever, опционально реранкирует, возвращает метрики."""
    public_questions = [
        {"id": str(row["public_id"]), "question": row["question"]}
        for row in qa_dataset["train"]
    ]
    results = {}
    for q in public_questions:
        raw = retriever.top_n(query=q["question"], n=retrieve_n)
        documents = raw["documents"][0]
        metadatas = raw["metadatas"][0]

        if reranker is not None:
            documents, metadatas, _ = reranker.rerank(
                query=q["question"],
                documents=documents,
                metadatas=metadatas,
                max_n=rerank_max_n,
            )

        found_ids = [int(m["id"]) for m in metadatas]
        results[q["id"]] = {"found_ids": found_ids, "model_answer": "-"}

    ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
    return ev_res["average_metrics"]["retrieval"]

## Бенчмарк

| Pipeline | Retrieve | Rerank |
|---|---|---|
| Semantic (baseline) | top-3 | — |
| BM25 | top-3 | — |
| Hybrid RRF | top-3 | — |
| Semantic + Reranker | top-10 → dynamic | cross-encoder |
| BM25 + Reranker | top-10 → dynamic | cross-encoder |
| Hybrid RRF + Reranker | top-10 → dynamic | cross-encoder |

In [7]:
print("1/6  Semantic baseline (top-3)...")
m_semantic = evaluate_retriever(semantic_retriever, qa_dataset, mapping, retrieve_n=3)
print(f"     {m_semantic}")

print("2/6  BM25 (top-3)...")
m_bm25 = evaluate_retriever(bm25_retriever, qa_dataset, mapping, retrieve_n=3)
print(f"     {m_bm25}")

print("3/6  Hybrid RRF (top-3)...")
m_hybrid = evaluate_retriever(hybrid_retriever, qa_dataset, mapping, retrieve_n=3)
print(f"     {m_hybrid}")

print("4/6  Semantic + Reranker (top-10 → dynamic)...")
m_semantic_rr = evaluate_retriever(semantic_retriever, qa_dataset, mapping, retrieve_n=10, reranker=reranker)
print(f"     {m_semantic_rr}")

print("5/6  BM25 + Reranker (top-10 → dynamic)...")
m_bm25_rr = evaluate_retriever(bm25_retriever, qa_dataset, mapping, retrieve_n=10, reranker=reranker)
print(f"     {m_bm25_rr}")

print("6/6  Hybrid RRF + Reranker (top-10 → dynamic)...")
m_hybrid_rr = evaluate_retriever(hybrid_retriever, qa_dataset, mapping, retrieve_n=10, reranker=reranker)
print(f"     {m_hybrid_rr}")

1/6  Semantic baseline (top-3)...
     {'hit_rate': np.float64(0.88), 'mrr': np.float64(0.8241666666666667), 'precision': np.float64(0.5138888888888888)}
2/6  BM25 (top-3)...
     {'hit_rate': np.float64(0.5166666666666667), 'mrr': np.float64(0.4666666666666667), 'precision': np.float64(0.21777777777777776)}
3/6  Hybrid RRF (top-3)...
     {'hit_rate': np.float64(0.8616666666666667), 'mrr': np.float64(0.735), 'precision': np.float64(0.4277777777777778)}
4/6  Semantic + Reranker (top-10 → dynamic)...
     {'hit_rate': np.float64(0.925), 'mrr': np.float64(0.8593611111111109), 'precision': np.float64(0.6209722222222223)}
5/6  BM25 + Reranker (top-10 → dynamic)...
     {'hit_rate': np.float64(0.6133333333333333), 'mrr': np.float64(0.5854722222222222), 'precision': np.float64(0.4395833333333333)}
6/6  Hybrid RRF + Reranker (top-10 → dynamic)...
     {'hit_rate': np.float64(0.9316666666666666), 'mrr': np.float64(0.876), 'precision': np.float64(0.6338055555555556)}


### Результаты

In [8]:
all_metrics = {
    "semantic (top-3)": m_semantic,
    "bm25 (top-3)": m_bm25,
    "hybrid RRF (top-3)": m_hybrid,
    "semantic + reranker": m_semantic_rr,
    "bm25 + reranker": m_bm25_rr,
    "hybrid RRF + reranker": m_hybrid_rr,
}

df = pd.DataFrame(all_metrics).T
df.index.name = "pipeline"
df = df[["hit_rate", "mrr", "precision"]].round(4)
df

,hit_rate,mrr,precision
pipeline,,,
semantic (top-3),0.8800,0.8242,0.5139
bm25 (top-3),0.5167,0.4667,0.2178
hybrid RRF (top-3),0.8617,0.7350,0.4278
semantic + reranker,0.9250,0.8594,0.6210
bm25 + reranker,0.6133,0.5855,0.4396
hybrid RRF + reranker,0.9317,0.8760,0.6338


In [ ]:
def to_serializable(d):
    return {k: float(v) for k, v in d.items()}

with open("../results/hybrid_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {name: to_serializable(metrics) for name, metrics in all_metrics.items()},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Результаты сохранены: results/hybrid_benchmark_results.json")